In [45]:
import pandas as pd
import os
import chardet
from thefuzz import process, fuzz

In [46]:
DATA_DIR = "F:/ranochaya/test 21vek,by/test"
CLIENTS_FILE = f"{DATA_DIR}/Клиенты.txt"
SALES_FILE = f"{DATA_DIR}/Продажи.txt"
CORRECTIONS_FILE = f"{DATA_DIR}/city_corrections.csv"
REVIEW_FILE = f"{DATA_DIR}/cities_to_review.csv"
CLEANED_FILE = f"{DATA_DIR}/cleaned_data.csv"

In [47]:
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

In [48]:
def normalize_city_name(city):

    if pd.isna(city):
        return ""
    return str(city).strip().lower()

def load_corrections():

    if os.path.exists(CORRECTIONS_FILE):
        df = pd.read_csv(CORRECTIONS_FILE)
        return dict(zip(df['Опечатка'], df['Правильный город']))
    return {}

def save_corrections(corrections):

    df = pd.DataFrame([
        {'Опечатка': k, 'Правильный город': v} 
        for k, v in corrections.items()
    ])
    df.to_csv(CORRECTIONS_FILE, index=False)

In [49]:
def auto_clean_cities():

    clients = pd.read_csv(CLIENTS_FILE, sep='\t', encoding='utf-8')
    sales = pd.read_csv(SALES_FILE, sep='\t', encoding='utf-8')
    
    correct_cities = sales['Город'].dropna().unique().tolist()
    correct_normalized = [normalize_city_name(city) for city in correct_cities]
    city_mapping = {normalize_city_name(city): city for city in correct_cities}
    
    print(f" Эталонных городов: {len(correct_cities)}")
    
    # Загружаем существующие исправления
    corrections = load_corrections()
    
    # Обрабатываем города клиентов
    unique_cities = clients['Город'].dropna().unique()
    need_review = []
    
    for city in unique_cities:
        original_city = str(city).strip()
        
        # Пропускаем если уже исправлено
        if original_city in corrections:
            continue
            
        normalized = normalize_city_name(city)
        if not normalized:
            continue
            
        # Ищем похожий город
        best_match, score = process.extractOne(normalized, correct_normalized, scorer=fuzz.token_sort_ratio)
        
        if score >= 90:  # Высокая уверенность: исправляем автоматически
            correct_city = city_mapping[best_match]
            corrections[original_city] = correct_city
            print(f" Авто: {original_city} → {correct_city}")
        else:
            # Добавляем в список для ручной проверки
            suggested = city_mapping.get(best_match, best_match)
            need_review.append({
                'Опечатка': original_city,
                'Предложенный_вариант': suggested,
                'Уверенность': score,
            })
            print(f" Проверить: {original_city} → {suggested} ({score}%)")

    save_corrections(corrections)
    
    # Файл для ручной проверки
    if need_review:
        review_df = pd.DataFrame(need_review)
        review_df['Правильный город'] = '' 
        review_df.to_csv(REVIEW_FILE, index=False, encoding="cp1251")
        print(f"Создан файл для проверки: {len(need_review)} городов")
    
    # Применяем исправления к данным
    clients['city_cleaned'] = clients['Город'].apply(
        lambda x: corrections.get(str(x).strip(), str(x).strip())
    )
    clients.to_csv(CLEANED_FILE, index=False)
    
    print(f"Готово! Обработано: {len(clients)} записей")

In [50]:
#Обработка ручных правок
def apply_manual_corrections():
    if not os.path.exists(REVIEW_FILE):
        print(" Файл для проверки не найден")
        return
    
    encoding = detect_encoding(REVIEW_FILE)
    review_df = pd.read_csv(REVIEW_FILE, sep=';', encoding=encoding)
    corrections = load_corrections()
    
    applied_count = 0
    
    for _, row in review_df.iterrows():
        wrong_city = str(row['Опечатка']).strip()
        user_correction = str(row['Правильный город']).strip()
        
        if user_correction and user_correction != 'nan':
            corrections[wrong_city] = user_correction
            applied_count += 1
            print(f"Исправлено: {wrong_city} → {user_correction}")
    
    if applied_count > 0:
        save_corrections(corrections)
        print(f"Применено исправлений: {applied_count}")
        
        # Обновляем очищенные данные
        clients = pd.read_csv(CLIENTS_FILE, sep='\t', encoding='utf-8')
        clients['city_cleaned'] = clients['Город'].apply(
            lambda x: corrections.get(str(x).strip(), str(x).strip())
        )
        clients.to_csv(CLEANED_FILE, index=False)

        archive_file = f"{DATA_DIR}/cities_to_review_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
        os.rename(REVIEW_FILE, archive_file)
        print(f"Файл перемещен в: {archive_file}")
    else:
        print("Нет новых исправлений")

In [51]:
auto_clean_cities()

 Эталонных городов: 7
Готово! Обработано: 5060 записей


In [53]:
apply_manual_corrections()

 Файл для проверки не найден


In [54]:
DATA_DIR = "F:/ranochaya/test 21vek,by/test"
RAW_FILE = pd.read_csv(f"{DATA_DIR}/Клиенты.txt", sep='\t', encoding='utf-8')
CORRECT_FILE =pd.read_csv(f"{DATA_DIR}/Продажи.txt", sep='\t', encoding='utf-8')
 
unique_cities = CORRECT_FILE['Город'].dropna().unique()
print(unique_cities)

['Санкт-Петербург' 'Казань' 'Ростов-на-Дону' 'Москва' 'Екатеринбург'
 'Самара' 'Новосибирск']
